# 00 — Data Preparation
### Shared preprocessing pipeline for all models
Run this notebook FIRST before any model notebook.

**Prepares data for:**
- Forecasting models (LSTM, GRU, TFT) — predicts future breakdown
- Classification model (FAG-TFT) — classifies current state
- Anomaly Gate — detects unknown faults

**Anti-overfitting strategy:**
- Time-series augmentation (jittering, scaling, magnitude warping) applied to TRAINING data only
- No augmented data in test set (prevents data leakage)
- Cited: Iwana & Uchida (2021, PLOS ONE) empirical survey of augmentation for time-series classification

In [15]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Imports complete.')

✅ Imports complete.


In [16]:
# FILE PATH
DATA_FILE = r'C:\Users\DeelakaD\OneDrive - MAS Holdings (Pvt) Ltd\Documents\GitHub\Preventing-Mechanism\Datasets\MBP_ControllerData_Augmented.xlsx'

TIME_STEPS = 5
H = 5  # Forecasting horizon: predict if breakdown in next 5 events

# Column name for machine state labels
STATE_COL = 'Machine States'

KNOWN_BREAKDOWNS = [
    'High thread tension',
    'High foot pressure',
    'Sewing without thread',
]
ALLOWED_STATES = ['Healthy'] + KNOWN_BREAKDOWNS

# AUGMENTATION CONFIG
AUG_TARGET_PER_CLASS = 500       # Target records per breakdown class (training only)
AUG_TARGET_HEALTHY   = 1000      # Target healthy records (training only)
JITTER_SIGMA         = 0.03      # Gaussian noise std (on scaled data)
SCALE_SIGMA          = 0.1       # Scaling factor std
WARP_SIGMA           = 0.2       # Magnitude warping std
WARP_KNOTS           = 4         # Number of spline knots for warping

# FREQUENCY BAND GROUPS (for FAG-TFT)
VIB_GROUP_1 = [f'{i}-{i+10}Hz' for i in range(10,  100, 10)]   # 10-100Hz
VIB_GROUP_2 = [f'{i}-{i+10}Hz' for i in range(100, 300, 10)]   # 100-300Hz
VIB_GROUP_3 = [f'{i}-{i+10}Hz' for i in range(300, 610, 10)]   # 300-610Hz
ELEC_FEATS  = ['machineVoltageMean','machineVoltageMin','machineVoltageMax',
               'machineCurrentMean','machineCurrentMin','machineCurrentMax']

print(f'✅ Config loaded.')
print(f'   TIME_STEPS             : {TIME_STEPS}')
print(f'   Forecasting horizon H  : {H}')
print(f'   State column           : {STATE_COL}')
print(f'   Known breakdowns       : {len(KNOWN_BREAKDOWNS)} — {KNOWN_BREAKDOWNS}')
print(f'   Augmentation targets   : {AUG_TARGET_HEALTHY} healthy, {AUG_TARGET_PER_CLASS} per breakdown')
print(f'   Augmentation methods   : Jittering(σ={JITTER_SIGMA}), Scaling(σ={SCALE_SIGMA}), MagnitudeWarping(σ={WARP_SIGMA}, knots={WARP_KNOTS})')


✅ Config loaded.
   TIME_STEPS             : 5
   Forecasting horizon H  : 5
   State column           : Machine States
   Known breakdowns       : 3 — ['High thread tension', 'High foot pressure', 'Sewing without thread']
   Augmentation targets   : 1000 healthy, 500 per breakdown
   Augmentation methods   : Jittering(σ=0.03), Scaling(σ=0.1), MagnitudeWarping(σ=0.2, knots=4)


In [17]:
# LOAD DATA
master_df = pd.read_excel(DATA_FILE)

# Filter valid vibration records (must start with '10')
master_df = master_df[master_df['machineVibration'].astype(str).str.startswith('10')].copy()

# Fill empty Machine States column as 'Healthy'
master_df[STATE_COL] = master_df[STATE_COL].fillna('Healthy')

# Keep only allowed states
master_df = master_df[master_df[STATE_COL].isin(ALLOWED_STATES)].reset_index(drop=True)

# Sort by dateTime (chronological order)
master_df['dateTime'] = pd.to_datetime(master_df['dateTime'])
master_df = master_df.sort_values('dateTime').reset_index(drop=True)

# Create time_gap: seconds between consecutive pedal presses (capped at 300s)
master_df['time_gap'] = master_df['dateTime'].diff().dt.total_seconds().fillna(0).clip(upper=300)

print(f'✅ Data loaded: {len(master_df)} total records')
print(f'   Date range: {master_df["dateTime"].iloc[0]} to {master_df["dateTime"].iloc[-1]}')
print(f'\nClass distribution:')
print(master_df[STATE_COL].value_counts())


✅ Data loaded: 2500 total records
   Date range: 2026-03-25 08:50:21 to 2026-03-26 11:29:49

Class distribution:
Machine States
Healthy                  1000
Sewing without thread     500
High thread tension       500
High foot pressure        500
Name: count, dtype: int64


In [18]:
# FEATURE EXTRACTION (67 features: 60 vib + 6 elec + 1 time_gap)
def extract_features(df):
    vib_records = []
    for val in df['machineVibration']:
        vib_dict = {}
        parts = str(val).split(',')
        try:
            for i in range(0, len(parts)-1, 2):
                f_start = int(parts[i])
                vib_dict[f'{f_start}-{f_start+10}Hz'] = int(parts[i+1])
        except: pass
        vib_records.append(vib_dict)

    expected_vib = [f'{i}-{i+10}Hz' for i in range(10, 610, 10)]
    vib_df  = pd.DataFrame(vib_records).reindex(columns=expected_vib, fill_value=0)
    elec_df = df[ELEC_FEATS].ffill().fillna(0).reset_index(drop=True)
    time_gap_df = df[['time_gap']].reset_index(drop=True)
    return pd.concat([vib_df.reset_index(drop=True), elec_df, time_gap_df], axis=1)

X_all = extract_features(master_df)
y_all = master_df[STATE_COL].values
print(f'✅ Features extracted: {X_all.shape}  (60 vib + 6 elec + 1 time_gap = 67 features)')


✅ Features extracted: (2500, 67)  (60 vib + 6 elec + 1 time_gap = 67 features)


---
## Classification Data (for FAG-TFT + Anomaly Gate)
### With time-series augmentation on training data only

In [19]:
# ENCODE LABELS FOR CLASSIFICATION
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_all)
num_classes = len(encoder.classes_)
print(f'✅ Classes: {list(encoder.classes_)}')

# TRAIN/TEST SPLIT (on REAL data only — no augmentation yet)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all.values, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print(f'\n   Train (before augmentation): {X_train_raw.shape}')
print(f'   Test  (never augmented)    : {X_test_raw.shape}')
print(f'   Train class counts: {np.bincount(y_train)}')
print(f'   Test  class counts: {np.bincount(y_test)}')

# ── TIME-SERIES AUGMENTATION (training data only) ──────────────────────
# Methods: Jittering, Scaling, Magnitude Warping
# Reference: Iwana & Uchida (2021), "An empirical survey of data augmentation
#            for time series classification with neural networks", PLOS ONE

from scipy.interpolate import CubicSpline

def augment_jitter(x, sigma=0.03):
    """Add independent Gaussian noise to each feature at each timestep."""
    return x + np.random.normal(0, sigma, x.shape)

def augment_scale(x, sigma=0.1):
    """Multiply entire sample by a random factor ~ N(1, sigma^2)."""
    factor = np.random.normal(1, sigma, size=(1, x.shape[1]))
    return x * factor

def augment_magnitude_warp(x, sigma=0.2, knots=4):
    """Apply smooth random curve as element-wise multiplier."""
    if x.shape[0] < 2:
        return x * np.random.normal(1, sigma, size=x.shape)
    orig_steps = np.arange(x.shape[0])
    random_steps = np.linspace(0, x.shape[0]-1, num=knots+2)
    random_values = np.random.normal(1, sigma, size=(knots+2, x.shape[1]))
    warp = np.zeros_like(x)
    for i in range(x.shape[1]):
        cs = CubicSpline(random_steps, random_values[:, i])
        warp[:, i] = cs(orig_steps)
    return x * warp

def augment_sample(x, jitter_sigma, scale_sigma, warp_sigma, warp_knots):
    """Apply a random combination of augmentations to one sample."""
    method = np.random.choice(['jitter', 'scale', 'warp', 'jitter+scale', 'jitter+warp'])
    if method == 'jitter':
        return augment_jitter(x, jitter_sigma)
    elif method == 'scale':
        return augment_scale(x, scale_sigma)
    elif method == 'warp':
        return augment_magnitude_warp(x, warp_sigma, warp_knots)
    elif method == 'jitter+scale':
        return augment_scale(augment_jitter(x, jitter_sigma), scale_sigma)
    elif method == 'jitter+warp':
        return augment_magnitude_warp(augment_jitter(x, jitter_sigma), warp_sigma, warp_knots)

# Determine augmentation targets per class
aug_X_list = [X_train_raw.copy()]
aug_y_list = [y_train.copy()]

for cls_idx in range(num_classes):
    cls_name = encoder.classes_[cls_idx]
    cls_mask = y_train == cls_idx
    cls_data = X_train_raw[cls_mask]
    current_count = len(cls_data)
    target = AUG_TARGET_HEALTHY if cls_name == 'Healthy' else AUG_TARGET_PER_CLASS
    needed = target - current_count

    if needed <= 0:
        print(f'  {cls_name}: {current_count} samples (no augmentation needed)')
        continue

    print(f'  {cls_name}: {current_count} real → generating {needed} augmented samples')
    aug_samples = []
    for _ in range(needed):
        # Pick a random real sample from this class
        src_idx = np.random.randint(0, current_count)
        src = cls_data[src_idx].reshape(1, -1)  # (1, 67)
        aug = augment_sample(src, JITTER_SIGMA, SCALE_SIGMA, WARP_SIGMA, WARP_KNOTS)
        aug_samples.append(aug.flatten())

    aug_X_list.append(np.array(aug_samples))
    aug_y_list.append(np.full(needed, cls_idx))

X_train_aug = np.concatenate(aug_X_list, axis=0)
y_train_aug = np.concatenate(aug_y_list, axis=0)

# Shuffle augmented training data
shuffle_idx = np.random.permutation(len(X_train_aug))
X_train_aug = X_train_aug[shuffle_idx]
y_train_aug = y_train_aug[shuffle_idx]

print(f'\n✅ Augmentation complete.')
print(f'   Train (after augmentation): {X_train_aug.shape}')
print(f'   Train class counts: {np.bincount(y_train_aug)}')
print(f'   Test  (unchanged)         : {X_test_raw.shape}')

# SCALE (fit on augmented training data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_aug)
X_test_scaled  = scaler.transform(X_test_raw)

# Update y_train to use augmented labels
y_train = y_train_aug

print(f'\n   Scaled train: {X_train_scaled.shape}  Scaled test: {X_test_scaled.shape}')


✅ Classes: ['Healthy', 'High foot pressure', 'High thread tension', 'Sewing without thread']

   Train (before augmentation): (2000, 67)
   Test  (never augmented)    : (500, 67)
   Train class counts: [800 400 400 400]
   Test  class counts: [200 100 100 100]
  Healthy: 800 real → generating 200 augmented samples
  High foot pressure: 400 real → generating 100 augmented samples
  High thread tension: 400 real → generating 100 augmented samples
  Sewing without thread: 400 real → generating 100 augmented samples

✅ Augmentation complete.
   Train (after augmentation): (2500, 67)
   Train class counts: [1000  500  500  500]
   Test  (unchanged)         : (500, 67)

   Scaled train: (2500, 67)  Scaled test: (500, 67)


In [20]:
# SEQUENCE CREATION FOR CLASSIFICATION
def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, TIME_STEPS)
X_test_seq,  y_test_seq  = create_sequences(X_test_scaled,  y_test,  TIME_STEPS)

print(f'✅ Classification sequences created.')
print(f'   Train: {X_train_seq.shape}  →  (samples, {TIME_STEPS}, 67)')
print(f'   Test : {X_test_seq.shape}')


✅ Classification sequences created.
   Train: (2495, 5, 67)  →  (samples, 5, 67)
   Test : (495, 5, 67)


In [21]:
# HEALTHY-ONLY DATA FOR ANOMALY GATE
healthy_mask = master_df[STATE_COL].values == 'Healthy'
X_healthy    = extract_features(master_df[healthy_mask])
X_healthy_scaled = scaler.transform(X_healthy.values)

def create_sequences_X(X, time_steps):
    return np.array([X[i:i+time_steps] for i in range(len(X)-time_steps)])

X_healthy_seq = create_sequences_X(X_healthy_scaled, TIME_STEPS)
split = int(len(X_healthy_seq)*0.8)
X_ae_train, X_ae_test = X_healthy_seq[:split], X_healthy_seq[split:]

print(f'✅ Autoencoder sequences: train={X_ae_train.shape}  test={X_ae_test.shape}')


✅ Autoencoder sequences: train=(796, 5, 67)  test=(199, 5, 67)


---
## Forecasting Data (for LSTM, GRU, TFT Forecaster)
### With time-series augmentation on training data only

In [22]:
# CREATE FORECASTING LABELS
# breakdown_now: is this record a breakdown?
master_df['breakdown_now'] = (master_df[STATE_COL] != 'Healthy').astype(int)

# future_breakdown: will a breakdown happen in the next H events?
# future_reason: which type of breakdown is coming?
future_breakdown = []
future_reason = []

for i in range(len(master_df)):
    future_window = master_df.iloc[i+1:i+1+H]
    bd_in_future = future_window[future_window[STATE_COL] != 'Healthy']
    if len(bd_in_future) > 0:
        future_breakdown.append(1)
        future_reason.append(bd_in_future.iloc[0][STATE_COL])
    else:
        future_breakdown.append(0)
        future_reason.append('No Breakdown')

master_df['future_breakdown'] = future_breakdown
master_df['future_reason'] = future_reason

# Remove rows where breakdown is already happening (can't forecast what already happened)
df_forecast = master_df[master_df['breakdown_now'] == 0].copy().reset_index(drop=True)

print(f'✅ Forecasting labels created.')
print(f'   Total records (active breakdowns removed): {len(df_forecast)}')
print(f'   Safe (no breakdown coming)   : {(df_forecast["future_breakdown"]==0).sum()}')
print(f'   Approaching breakdown        : {(df_forecast["future_breakdown"]==1).sum()}')
print(f'\n   Approaching breakdown by type:')
print(df_forecast[df_forecast['future_breakdown']==1]['future_reason'].value_counts().to_string())


✅ Forecasting labels created.
   Total records (active breakdowns removed): 1000
   Safe (no breakdown coming)   : 970
   Approaching breakdown        : 30

   Approaching breakdown by type:
future_reason
Sewing without thread    25
High thread tension       5


In [23]:
# PREPARE FORECASTING SEQUENCES
X_forecast = extract_features(df_forecast)
y_forecast_binary = df_forecast['future_breakdown'].values

# Encode breakdown reason for TFT type prediction
encoder_reason = LabelEncoder()
y_forecast_reason = encoder_reason.fit_transform(df_forecast['future_reason'].values)
num_reason_classes = len(encoder_reason.classes_)

# Train/test split for forecasting (REAL data only)
X_fc_train_raw, X_fc_test_raw, y_fc_train_bin, y_fc_test_bin, y_fc_train_type, y_fc_test_type = train_test_split(
    X_forecast.values, y_forecast_binary, y_forecast_reason,
    test_size=0.2, random_state=42, stratify=y_forecast_binary)

# ── AUGMENT FORECASTING TRAINING DATA ──────────────────────────────────
print(f'Forecasting train BEFORE augmentation: {X_fc_train_raw.shape}')
print(f'  Binary class counts: {np.bincount(y_fc_train_bin)}')

# Augment minority class (breakdown approaching = 1)
fc_aug_X = [X_fc_train_raw.copy()]
fc_aug_bin = [y_fc_train_bin.copy()]
fc_aug_type = [y_fc_train_type.copy()]

minority_mask = y_fc_train_bin == 1
minority_X = X_fc_train_raw[minority_mask]
minority_type = y_fc_train_type[minority_mask]
majority_count = (y_fc_train_bin == 0).sum()
minority_count = minority_mask.sum()
needed = majority_count - minority_count  # Balance to majority count

if needed > 0:
    print(f'  Augmenting breakdown-approaching class: {minority_count} → {majority_count}')
    aug_samples = []
    aug_types = []
    for _ in range(needed):
        src_idx = np.random.randint(0, minority_count)
        src = minority_X[src_idx].reshape(1, -1)
        aug = augment_sample(src, JITTER_SIGMA, SCALE_SIGMA, WARP_SIGMA, WARP_KNOTS)
        aug_samples.append(aug.flatten())
        aug_types.append(minority_type[src_idx])
    fc_aug_X.append(np.array(aug_samples))
    fc_aug_bin.append(np.ones(needed, dtype=int))
    fc_aug_type.append(np.array(aug_types))

X_fc_train_aug = np.concatenate(fc_aug_X, axis=0)
y_fc_train_bin_aug = np.concatenate(fc_aug_bin, axis=0)
y_fc_train_type_aug = np.concatenate(fc_aug_type, axis=0)

# Shuffle
fc_shuf = np.random.permutation(len(X_fc_train_aug))
X_fc_train_aug = X_fc_train_aug[fc_shuf]
y_fc_train_bin_aug = y_fc_train_bin_aug[fc_shuf]
y_fc_train_type_aug = y_fc_train_type_aug[fc_shuf]

print(f'Forecasting train AFTER augmentation: {X_fc_train_aug.shape}')
print(f'  Binary class counts: {np.bincount(y_fc_train_bin_aug)}')

# Scale using SAME scaler as classification (consistency across all models)
X_fc_train_scaled = scaler.transform(X_fc_train_aug)
X_fc_test_scaled  = scaler.transform(X_fc_test_raw)

# Create sequences
X_fc_train_seq, y_fc_train_bin_seq = create_sequences(X_fc_train_scaled, y_fc_train_bin_aug, TIME_STEPS)
X_fc_test_seq,  y_fc_test_bin_seq  = create_sequences(X_fc_test_scaled,  y_fc_test_bin,  TIME_STEPS)

# Type sequences (same X, different y)
_, y_fc_train_type_seq = create_sequences(X_fc_train_scaled, y_fc_train_type_aug, TIME_STEPS)
_, y_fc_test_type_seq  = create_sequences(X_fc_test_scaled,  y_fc_test_type,  TIME_STEPS)

print(f'\n✅ Forecasting sequences created.')
print(f'   Train: {X_fc_train_seq.shape}')
print(f'   Test : {X_fc_test_seq.shape}')
print(f'   Binary class counts (train seq): {np.bincount(y_fc_train_bin_seq)}')
print(f'   Reason classes: {list(encoder_reason.classes_)}')


Forecasting train BEFORE augmentation: (800, 67)
  Binary class counts: [776  24]
  Augmenting breakdown-approaching class: 24 → 776
Forecasting train AFTER augmentation: (1552, 67)
  Binary class counts: [776 776]

✅ Forecasting sequences created.
   Train: (1547, 5, 67)
   Test : (195, 5, 67)
   Binary class counts (train seq): [774 773]
   Reason classes: ['High thread tension', 'No Breakdown', 'Sewing without thread']


---
## Save All Prepared Data


In [24]:
# SAVE CLASSIFICATION DATA
prepared_classification = {
    'X_train_seq'  : X_train_seq,
    'X_test_seq'   : X_test_seq,
    'y_train_seq'  : y_train_seq,
    'y_test_seq'   : y_test_seq,
    'X_ae_train'   : X_ae_train,
    'X_ae_test'    : X_ae_test,
    'num_classes'  : num_classes,
    'num_features' : X_train_seq.shape[2],
    'TIME_STEPS'   : TIME_STEPS,
    'VIB_GROUP_1'  : VIB_GROUP_1,
    'VIB_GROUP_2'  : VIB_GROUP_2,
    'VIB_GROUP_3'  : VIB_GROUP_3,
    'ELEC_FEATS'   : ELEC_FEATS,
    'feature_names': list(X_all.columns),
}

# SAVE FORECASTING DATA
prepared_forecasting = {
    'X_fc_train_seq'      : X_fc_train_seq,
    'X_fc_test_seq'       : X_fc_test_seq,
    'y_fc_train_bin_seq'  : y_fc_train_bin_seq,
    'y_fc_test_bin_seq'   : y_fc_test_bin_seq,
    'y_fc_train_type_seq' : y_fc_train_type_seq,
    'y_fc_test_type_seq'  : y_fc_test_type_seq,
    'num_features'        : X_fc_train_seq.shape[2],
    'num_reason_classes'  : num_reason_classes,
    'TIME_STEPS'          : TIME_STEPS,
    'H'                   : H,
    'feature_names'       : list(X_forecast.columns),
}

with open('prepared_classification.pkl','wb') as f: pickle.dump(prepared_classification, f)
with open('prepared_forecasting.pkl','wb') as f: pickle.dump(prepared_forecasting, f)  # for LSTM, GRU, CNN LSTM(training and testing sequences)
with open('scaler.pkl','wb') as f: pickle.dump(scaler, f) # this carrries training mean and std for Live Analyser file
with open('encoder.pkl','wb') as f: pickle.dump(encoder, f) # this carries the mapping of class labels to integers for Live Analyser file
with open('encoder_reason.pkl','wb') as f: pickle.dump(encoder_reason, f)

print('✅ Saved:')
print('   prepared_classification.pkl  — for FAG-TFT + Anomaly Gate')
print('   prepared_forecasting.pkl     — for LSTM, GRU, TFT Forecaster')
print('   scaler.pkl                   — shared scaler')
print('   encoder.pkl                  — classification label encoder')
print('   encoder_reason.pkl           — forecasting reason encoder')
print(f'\n   Classification: {num_classes} classes, {X_train_seq.shape[2]} features, TIME_STEPS={TIME_STEPS}')
print(f'   Forecasting: binary + {num_reason_classes} reason classes, {X_fc_train_seq.shape[2]} features, H={H}')


✅ Saved:
   prepared_classification.pkl  — for FAG-TFT + Anomaly Gate
   prepared_forecasting.pkl     — for LSTM, GRU, TFT Forecaster
   scaler.pkl                   — shared scaler
   encoder.pkl                  — classification label encoder
   encoder_reason.pkl           — forecasting reason encoder

   Classification: 4 classes, 67 features, TIME_STEPS=5
   Forecasting: binary + 3 reason classes, 67 features, H=5
